# 05 Train phrase GRU
Upload `phrases.zip` from `data/custom/phrases`. Split by session prefix when possible. Sequence augmentation is in `train_utils.augment_sequence`.


In [ ]:
# 1) Mount Google Drive (a popup asks permission)
from google.colab import drive
drive.mount("/content/drive")

# 2) Paths
DRIVE = "/content/drive/MyDrive/sign-language"

# 3) Code + packages
!git clone https://github.com/<you>/sign-language-translator.git /content/repo 2>/dev/null || git -C /content/repo pull
!pip install -q mediapipe==0.10.14 "numpy<2" scikit-learn seaborn tqdm
%cd /content/repo
import sys
sys.path.insert(0, "/content/repo")



In [ ]:
!cp {DRIVE}/custom/phrases.zip /content/ && unzip -q /content/phrases.zip -d /content/phrases
import glob, json, os
from pathlib import Path
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

from src.config import PHRASE_LABELS, SEQ_LEN, N_FEATURES
from src.models import build_sequence_model
from src.train_utils import (
    class_weights_if_imbalanced, default_callbacks, plot_confusion_matrix, plot_history, sequence_dataset,
)

X, y, sessions = [], [], []
for i, lab in enumerate(PHRASE_LABELS):
    for f in glob.glob(f"/content/phrases/**/{lab}/*.npy", recursive=True):
        arr = np.load(f)
        if arr.shape != (SEQ_LEN, N_FEATURES):
            print("skip bad shape", f, arr.shape)
            continue
        X.append(arr)
        y.append(i)
        sessions.append(Path(f).stem.split("_")[0])
X, y = np.array(X, dtype=np.float32), np.array(y)
print(X.shape, np.bincount(y), "sessions", set(sessions))

# split by session if more than one session id, else random 70/15/15
uniq = sorted(set(sessions))
if len(uniq) >= 3:
    val_s, test_s = uniq[-2], uniq[-1]
    train_idx = [i for i, s in enumerate(sessions) if s not in (val_s, test_s)]
    val_idx = [i for i, s in enumerate(sessions) if s == val_s]
    test_idx = [i for i, s in enumerate(sessions) if s == test_s]
    Xtr, Xva, Xte = X[train_idx], X[val_idx], X[test_idx]
    ytr, yva, yte = y[train_idx], y[val_idx], y[test_idx]
else:
    Xtr, Xtmp, ytr, ytmp = train_test_split(X, y, test_size=0.30, random_state=0, stratify=y)
    Xva, Xte, yva, yte = train_test_split(Xtmp, ytmp, test_size=0.50, random_state=0, stratify=ytmp)

cw = class_weights_if_imbalanced(ytr)
train_ds = sequence_dataset(Xtr, ytr, batch_size=32, augment=True)
val_ds = sequence_dataset(Xva, yva, batch_size=32, augment=False, shuffle=False)
test_ds = sequence_dataset(Xte, yte, batch_size=32, augment=False, shuffle=False)

model = build_sequence_model(len(PHRASE_LABELS), SEQ_LEN, N_FEATURES)
model.compile(tf.keras.optimizers.Adam(1e-3), "sparse_categorical_crossentropy", metrics=["accuracy"])
cbs = default_callbacks(f"{DRIVE}/models/phrases_gru.keras", patience=12)
history = model.fit(train_ds, validation_data=val_ds, epochs=100, callbacks=cbs, class_weight=cw)

Path(f"{DRIVE}/reports").mkdir(parents=True, exist_ok=True)
plot_history(history, f"{DRIVE}/reports/history_gru.png")
y_pred = np.argmax(model.predict(test_ds), axis=1)
print(classification_report(yte, y_pred, target_names=PHRASE_LABELS))
plot_confusion_matrix(yte, y_pred, PHRASE_LABELS, f"{DRIVE}/reports/cm_gru.png")
print("params", model.count_params())
from src.config import ALPHABET_LABELS
with open(f"{DRIVE}/models/labels.json", "w") as f:
    json.dump({"alphabet": ALPHABET_LABELS, "phrases": PHRASE_LABELS}, f)
model.save(f"{DRIVE}/models/phrases_gru.keras")

